# Dynamic Transformer Extraction Prototype

This notebook extracts the same SQL-ready fields as the first `042-2024/25` prototype, but the indicator extraction no longer assumes fixed Excel coordinates.

Instead it dynamically detects:

1. the best report sheet,
2. the Soll/Ist header row or rows,
3. the indicator description column,
4. the target indicator row via transformer embeddings,
5. the Soll/Ist/Erfüllung columns via transformer embeddings plus light lexical guards,
6. the exact values from the row/column intersections.

The notebook still defaults to project `042-2024/25`, but the table detector is designed to also handle older layouts such as the 2019 `Indikatorenbericht` format.

## 0. Imports and project files

In [1]:
from pathlib import Path
import json
import re
from datetime import datetime

import pandas as pd
from openpyxl import load_workbook
from docx import Document
from IPython.display import Markdown, display

from sentence_transformers import SentenceTransformer, util
from gliner import GLiNER

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "Hackathon" / "2024_2025"

PROJECT_DESCRIPTION_DOCX = DATA_DIR / "042_HandWerk.Zukunft_2024_2025_Projektbeschreibung.docx"
FINAL_REPORT_DOCX = DATA_DIR / "042_HandWerk.Zukunft-2024_2025_Inhaltlicher Endbericht.docx"
SOLL_XLSX = DATA_DIR / "042_HandWerk.Zukunft_2024_25_Indikatorenblatt.xlsx"
IST_XLSX = DATA_DIR / "042_HandWerk.Zukunft-2024-2025_Indikatorenbericht.xlsx"

for path in [PROJECT_DESCRIPTION_DOCX, FINAL_REPORT_DOCX, SOLL_XLSX, IST_XLSX]:
    assert path.exists(), f"Missing file: {path}"

print("Using files:")
for path in [PROJECT_DESCRIPTION_DOCX, FINAL_REPORT_DOCX, SOLL_XLSX, IST_XLSX]:
    print("-", path.relative_to(ROOT))

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using files:
- Hackathon/2024_2025/042_HandWerk.Zukunft_2024_2025_Projektbeschreibung.docx
- Hackathon/2024_2025/042_HandWerk.Zukunft-2024_2025_Inhaltlicher Endbericht.docx
- Hackathon/2024_2025/042_HandWerk.Zukunft_2024_25_Indikatorenblatt.xlsx
- Hackathon/2024_2025/042_HandWerk.Zukunft-2024-2025_Indikatorenbericht.xlsx


## 1. Load transformer models

Transformers used:

- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` for German semantic label matching in Excel and Word evidence selection.
- `urchade/gliner_multi-v2.1` for German open-vocabulary NER on Word text.

In [2]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GLINER_MODEL_NAME = "urchade/gliner_multi-v2.1"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME)

print("Loaded:", EMBEDDING_MODEL_NAME)
print("Loaded:", GLINER_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 236.63it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 19840.61it/s]


Loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Loaded: urchade/gliner_multi-v2.1


## 2. Shared helper functions

In [3]:
def clean_text(value):
    if value is None:
        return ""
    if isinstance(value, datetime):
        return value.date().isoformat()
    text = str(value).replace("\n", " ").strip()
    return re.sub(r"\s+", " ", text)


def normalize_for_match(text):
    return clean_text(text).lower().replace("/", " ").replace("-", " ")


def parse_excel_date(value):
    if isinstance(value, datetime):
        return value.date().isoformat()
    text = clean_text(value)
    if not text:
        return None
    if "00:00:00" in text:
        return text.split()[0]
    return text


def numeric_or_none(value):
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    text = clean_text(value).replace("%", "").replace(",", ".")
    if text in {"#DIV/0!", "#VALUE!", "#N/A"}:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def cosine_scores(query, candidates):
    query_embedding = embedding_model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    candidate_embeddings = embedding_model.encode(candidates, convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(query_embedding, candidate_embeddings)[0]


def best_semantic_match(query, candidates, min_score=0.45):
    if not candidates:
        return None
    scores = cosine_scores(query, candidates)
    best_idx = int(scores.argmax())
    best_score = float(scores[best_idx])
    if best_score < min_score:
        return None
    return best_idx, best_score


def format_number(value):
    if value is None:
        return None
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value

## 3. Dynamic Excel metadata extraction

For key-value metadata, the label and value usually sit on the same row. We search all sheets for the best German label match, then read the nearest non-empty value to the right.

In [4]:
def non_empty_cells(ws, max_row=None, max_col=None):
    max_row = min(max_row or ws.max_row, ws.max_row)
    max_col = min(max_col or ws.max_column, ws.max_column)
    cells = []
    for row in ws.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col):
        for cell in row:
            text = clean_text(cell.value)
            if text:
                cells.append({
                    "sheet": ws.title,
                    "row": cell.row,
                    "col": cell.column,
                    "coordinate": cell.coordinate,
                    "text": text,
                    "value": cell.value,
                })
    return cells


def nearest_right_value(ws, row, col, max_steps=10):
    for next_col in range(col + 1, min(ws.max_column, col + max_steps) + 1):
        cell = ws.cell(row=row, column=next_col)
        if clean_text(cell.value):
            return cell.value, cell.coordinate
    return None, None


def extract_key_value_semantic_any_sheet(workbook_path, german_query, max_row=50, max_col=25, min_score=0.48):
    wb = load_workbook(workbook_path, data_only=True, read_only=False)
    all_cells = []
    for ws in wb.worksheets:
        all_cells.extend(non_empty_cells(ws, max_row=max_row, max_col=max_col))
    texts = [cell["text"] for cell in all_cells]
    best = best_semantic_match(german_query, texts, min_score=min_score)
    if best is None:
        return {"value": None, "confidence": 0.0, "evidence": None}

    idx, score = best
    label_cell = all_cells[idx]
    ws = wb[label_cell["sheet"]]
    value, value_coordinate = nearest_right_value(ws, label_cell["row"], label_cell["col"])
    return {
        "value": clean_text(value),
        "confidence": round(score, 3),
        "evidence": {
            "source_file": str(workbook_path.relative_to(ROOT)),
            "sheet": label_cell["sheet"],
            "matched_label": label_cell["text"],
            "label_cell": label_cell["coordinate"],
            "value_cell": value_coordinate,
        },
    }


excel_fields = {
    "projekttraeger": "Projektträger oder Trägerorganisation",
    "projekttitel": "Projekttitel oder Projektname",
    "projektnummer": "Projektnummer oder Geschäftszeichen",
    "laufzeit_beginn": "Laufzeit Beginn oder Projektstart",
    "laufzeit_ende": "Laufzeit Ende oder Projektende",
    "hauptprojektstandort": "Hauptprojektstandort oder Bundesland des Projekts",
}

project_meta = {
    field: extract_key_value_semantic_any_sheet(IST_XLSX, query)
    for field, query in excel_fields.items()
}
project_meta["laufzeit_beginn"]["value"] = parse_excel_date(project_meta["laufzeit_beginn"]["value"])
project_meta["laufzeit_ende"]["value"] = parse_excel_date(project_meta["laufzeit_ende"]["value"])

pd.DataFrame([
    {
        "field": key,
        "value": item["value"],
        "confidence": item["confidence"],
        "sheet": item["evidence"]["sheet"] if item["evidence"] else None,
        "matched_label": item["evidence"]["matched_label"] if item["evidence"] else None,
        "value_cell": item["evidence"]["value_cell"] if item["evidence"] else None,
    }
    for key, item in project_meta.items()
])

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,field,value,confidence,sheet,matched_label,value_cell
0,projekttraeger,,0.764,1. ZWB,Angaben zum Projekt,NaN
1,projekttitel,HandWerk.Zukunft – Berufsvorbereitungskurse mi...,0.697,Overview IB,Projekttitel,D5
2,projektnummer,042-2024/25,0.844,Overview IB,Projektnummer,D6
3,laufzeit_beginn,2024-01-01,0.818,Overview IB,Laufzeit Beginn,D9
4,laufzeit_ende,2025-12-31,0.794,Overview IB,Laufzeit Ende,D10
5,hauptprojektstandort,Steiermark,0.901,Overview IB,Hauptprojektstandort,D8


## 4. Dynamic indicator table detection

This is the main upgrade over the first notebook. The old version assumed the 2024 layout. This version discovers the layout from cell content.

In [5]:
HEADER_KEYWORDS = re.compile(r"\b(soll|ist|erfüllt|erfuellt|vertrag|endbericht|berichtigt|begründung|begruendung|anmerkung)\b", re.I)
REPORT_KEYWORDS = re.compile(r"(indikatorenbericht|endbericht indikatoren|zielindikatoren|soll|ist|erfüllt|erfuellt)", re.I)
INDICATOR_HINTS = re.compile(r"(anzahl|teilnehmer|teilnehm|projektteilnehm|veranstaltung|beratung|kurs|workshop|stunden|standorte)", re.I)


def row_values(ws, row, max_col=None):
    max_col = min(max_col or ws.max_column, ws.max_column)
    return [clean_text(ws.cell(row=row, column=col).value) for col in range(1, max_col + 1)]


def score_report_sheet(ws):
    title = ws.title.strip().lower()
    title_boost = 0
    # Prefer the final-report sheet when modern workbooks contain both an overview sheet
    # and period sheets. Older 2019 workbooks usually only have one sheet.
    if title in {"eb", "endbericht"} or "endbericht" in title:
        title_boost += 100
    elif "zwb" in title:
        title_boost += 20
    elif "overview" in title:
        title_boost -= 10
    text_parts = []
    for row in range(1, min(ws.max_row, 80) + 1):
        text_parts.extend(row_values(ws, row, max_col=min(ws.max_column, 40)))
    text = " ".join(text_parts)
    return title_boost + len(REPORT_KEYWORDS.findall(text))


def score_header_row(ws, row):
    values = row_values(ws, row, max_col=min(ws.max_column, 80))
    keyword_hits = sum(1 for value in values if HEADER_KEYWORDS.search(value))
    soll_ist_balance = int(any("soll" in value.lower() for value in values)) + int(any("ist" in value.lower() for value in values))
    percent_hit = int(any("erfüllt" in value.lower() or "erfuellt" in value.lower() for value in values))
    return keyword_hits + soll_ist_balance + percent_hit


def detect_header_rows(ws):
    scored = [(row, score_header_row(ws, row)) for row in range(1, min(ws.max_row, 120) + 1)]
    best_row, best_score = max(scored, key=lambda item: item[1])
    if best_score < 2:
        raise ValueError(f"Could not detect Soll/Ist header row in sheet {ws.title}")

    header_rows = [best_row]
    next_values = row_values(ws, best_row + 1, max_col=min(ws.max_column, 80)) if best_row < ws.max_row else []
    next_text = " ".join(next_values)
    if re.search(r"\bM\s*\d+\b", next_text) or score_header_row(ws, best_row + 1) >= 2:
        header_rows.append(best_row + 1)
    return header_rows


def detect_indicator_column(ws, header_rows):
    data_start = max(header_rows) + 1
    col_scores = []
    for col in range(1, min(ws.max_column, 40) + 1):
        score = 0
        for row in range(data_start, min(ws.max_row, data_start + 100) + 1):
            text = clean_text(ws.cell(row=row, column=col).value)
            if len(text) >= 12 and numeric_or_none(text) is None:
                score += 1
                if INDICATOR_HINTS.search(text):
                    score += 2
                if text.lower().startswith("bereich"):
                    score += 1
        col_scores.append((col, score))
    best_col, best_score = max(col_scores, key=lambda item: item[1])
    if best_score == 0:
        raise ValueError(f"Could not detect indicator label column in sheet {ws.title}")
    return best_col


def build_column_headers(ws, header_rows):
    headers = {}
    for col in range(1, ws.max_column + 1):
        parts = [clean_text(ws.cell(row=row, column=col).value) for row in header_rows]
        header = re.sub(r"\s+", " ", " ".join(part for part in parts if part)).strip()
        if header:
            headers[col] = header
    return headers


def find_local_header_rows(ws, indicator_row, global_header_rows):
    candidates = []
    for row in range(max(1, indicator_row - 8), indicator_row):
        score = score_header_row(ws, row)
        if score >= 2:
            candidates.append((row, score))
    if not candidates:
        return global_header_rows
    best_row, _ = max(candidates, key=lambda item: item[1])
    header_rows = [best_row]
    if best_row + 1 < indicator_row:
        next_text = " ".join(row_values(ws, best_row + 1, max_col=min(ws.max_column, 80)))
        if re.search(r"\bM\s*\d+\b", next_text) or score_header_row(ws, best_row + 1) >= 2:
            header_rows.append(best_row + 1)
    return header_rows


def detect_indicator_table(workbook_path):
    wb = load_workbook(workbook_path, data_only=True, read_only=False)
    sheets_by_score = sorted(
        [(ws.title, score_report_sheet(ws)) for ws in wb.worksheets],
        key=lambda item: item[1],
        reverse=True,
    )
    sheet_name = sheets_by_score[0][0]
    ws = wb[sheet_name]
    header_rows = detect_header_rows(ws)
    indicator_col = detect_indicator_column(ws, header_rows)
    return {
        "workbook": wb,
        "worksheet": ws,
        "sheet_name": sheet_name,
        "sheet_scores": sheets_by_score,
        "header_rows": header_rows,
        "indicator_col": indicator_col,
        "data_start": max(header_rows) + 1,
    }


detected_table = detect_indicator_table(IST_XLSX)
detected_table_summary = {
    "sheet_name": detected_table["sheet_name"],
    "top_sheet_scores": detected_table["sheet_scores"][:5],
    "header_rows": detected_table["header_rows"],
    "indicator_col": detected_table["indicator_col"],
    "data_start": detected_table["data_start"],
}
detected_table_summary

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


{'sheet_name': 'EB',
 'top_sheet_scores': [('EB', 139),
  ('1. ZWB', 62),
  ('2. ZWB', 59),
  ('3. ZWB', 59),
  ('Overview IB', 54)],
 'header_rows': [9, 10],
 'indicator_col': 3,
 'data_start': 11}

## 5. Extract the main Soll/Ist KPI dynamically

The extraction now works by semantic row matching plus semantic column matching. Once row and columns are identified, the values are read from exact Excel coordinates.

In [6]:
def column_match_score(query, header, required_terms=None, forbidden_terms=None):
    base_score = float(cosine_scores(query, [header])[0])
    text = normalize_for_match(header)
    score = base_score
    for term in required_terms or []:
        if term in text:
            score += 0.12
    for term in forbidden_terms or []:
        if term in text:
            score -= 0.18
    return score


def best_column(headers, query, required_terms=None, forbidden_terms=None, min_score=0.35):
    scored = []
    for col, header in headers.items():
        score = column_match_score(query, header, required_terms=required_terms, forbidden_terms=forbidden_terms)
        scored.append((col, header, score))
    best = max(scored, key=lambda item: item[2])
    if best[2] < min_score:
        raise ValueError(f"Could not match column for query: {query}")
    return best


def extract_indicator_kpi_dynamic(workbook_path, indicator_query="Anzahl der Projektteilnehmenden gesamt"):
    table = detect_indicator_table(workbook_path)
    ws = table["worksheet"]
    indicator_col = table["indicator_col"]

    row_candidates = []
    for row in range(table["data_start"], ws.max_row + 1):
        label = clean_text(ws.cell(row=row, column=indicator_col).value)
        if label and len(label) > 6 and not label.lower().startswith("bereich"):
            row_candidates.append((row, label))

    best_row = best_semantic_match(indicator_query, [label for _, label in row_candidates], min_score=0.45)
    if best_row is None:
        raise ValueError(f"Could not find indicator row for query: {indicator_query}")

    row_idx, row_score = best_row
    indicator_row, matched_indicator = row_candidates[row_idx]

    local_header_rows = find_local_header_rows(ws, indicator_row, table["header_rows"])
    headers = build_column_headers(ws, local_header_rows)

    soll_col, soll_header, soll_score = best_column(
        headers,
        "Anzahl Soll laut Vertrag",
        required_terms=["soll"],
        forbidden_terms=["maßnahme", "massnahme", " m 1", " m1"],
    )
    ist_col, ist_header, ist_score = best_column(
        headers,
        "Anzahl Ist Endbericht gesamt erreicht",
        required_terms=["ist"],
        forbidden_terms=["maßnahme", "massnahme", " m 1", " m1"],
    )
    percent_col, percent_header, percent_score = best_column(
        headers,
        "Erfüllt in Prozent Zielerreichung",
        required_terms=["erfüllt"],
        forbidden_terms=[],
    )

    soll = numeric_or_none(ws.cell(row=indicator_row, column=soll_col).value)
    ist = numeric_or_none(ws.cell(row=indicator_row, column=ist_col).value)
    fulfillment = numeric_or_none(ws.cell(row=indicator_row, column=percent_col).value)
    if fulfillment is not None and fulfillment <= 2:
        fulfillment_percent = fulfillment * 100
    else:
        fulfillment_percent = fulfillment

    return {
        "indikator": matched_indicator,
        "soll": format_number(soll),
        "ist": format_number(ist),
        "erfuellung_prozent": round(fulfillment_percent, 1) if fulfillment_percent is not None else None,
        "confidence": round(min(row_score, soll_score, ist_score, percent_score), 3),
        "evidence": {
            "source_file": str(workbook_path.relative_to(ROOT)),
            "sheet": table["sheet_name"],
            "global_header_rows": table["header_rows"],
            "local_header_rows": local_header_rows,
            "indicator_col": indicator_col,
            "indicator_row": indicator_row,
            "matched_indicator": matched_indicator,
            "soll_header": soll_header,
            "soll_cell": ws.cell(row=indicator_row, column=soll_col).coordinate,
            "ist_header": ist_header,
            "ist_cell": ws.cell(row=indicator_row, column=ist_col).coordinate,
            "percent_header": percent_header,
            "percent_cell": ws.cell(row=indicator_row, column=percent_col).coordinate,
        },
    }


main_kpi = extract_indicator_kpi_dynamic(IST_XLSX)
main_kpi

{'indikator': 'Anzahl der Projektteilnehmenden gesamt',
 'soll': 180,
 'ist': None,
 'erfuellung_prozent': 107.8,
 'confidence': 0.763,
 'evidence': {'source_file': 'Hackathon/2024_2025/042_HandWerk.Zukunft-2024-2025_Indikatorenbericht.xlsx',
  'sheet': 'EB',
  'global_header_rows': [9, 10],
  'local_header_rows': [9, 10],
  'indicator_col': 3,
  'indicator_row': 11,
  'matched_indicator': 'Anzahl der Projektteilnehmenden gesamt',
  'soll_header': 'Anzahl SOLL lt. Vertrag',
  'soll_cell': 'D11',
  'ist_header': 'Anzahl IST - gesamt - berichtigt',
  'ist_cell': 'R11',
  'percent_header': 'Erfüllt in %',
  'percent_cell': 'S11'}}

## 6. Optional sanity check against a 2019-style report

This cell demonstrates the dynamic detector on a different layout. It is optional and does not feed the final SQL row for project `042-2024/25`.

In [7]:
OPTIONAL_2019_REPORT = ROOT / "Hackathon" / "2019_2020" / "035_DEUTSCH-IM-ALLTAG_2019_2020_Indikatorenbericht.xlsx"

if OPTIONAL_2019_REPORT.exists():
    detected_2019 = detect_indicator_table(OPTIONAL_2019_REPORT)
    demo_2019 = extract_indicator_kpi_dynamic(
        OPTIONAL_2019_REPORT,
        indicator_query="Anzahl der Projektteilnehmenden gesamt",
    )
    display({
        "detected_sheet": detected_2019["sheet_name"],
        "header_rows": detected_2019["header_rows"],
        "indicator_col": detected_2019["indicator_col"],
        "extracted_kpi": demo_2019,
    })
else:
    print("Optional 2019 report not found.")

{'detected_sheet': 'Indikatoren',
 'header_rows': [7],
 'indicator_col': 3,
 'extracted_kpi': {'indikator': 'Anzahl der Teilnehmerinnen und Teilnehmer im Projekt',
  'soll': 80,
  'ist': 76,
  'erfuellung_prozent': 95.0,
  'confidence': 0.832,
  'evidence': {'source_file': 'Hackathon/2019_2020/035_DEUTSCH-IM-ALLTAG_2019_2020_Indikatorenbericht.xlsx',
   'sheet': 'Indikatoren',
   'global_header_rows': [7],
   'local_header_rows': [7],
   'indicator_col': 3,
   'indicator_row': 9,
   'matched_indicator': 'Anzahl der Teilnehmerinnen und Teilnehmer im Projekt',
   'soll_header': 'Anzahl lt. Vertrag (SOLL)',
   'soll_cell': 'D9',
   'ist_header': 'Anzahl Endbericht (IST)',
   'ist_cell': 'E9',
   'percent_header': 'Erfüllt in %',
   'percent_cell': 'F9'}}}

## 7. Word parsing and GLiNER extraction

In [8]:
def docx_text_blocks(path):
    doc = Document(path)
    blocks = []

    for idx, para in enumerate(doc.paragraphs):
        text = clean_text(para.text)
        if text:
            blocks.append({
                "source_file": str(path.relative_to(ROOT)),
                "kind": "paragraph",
                "index": idx,
                "style": para.style.name,
                "text": text,
            })

    for table_idx, table in enumerate(doc.tables):
        cell_texts = []
        for row in table.rows:
            for cell in row.cells:
                text = clean_text(cell.text)
                if text:
                    cell_texts.append(text)
        if cell_texts:
            blocks.append({
                "source_file": str(path.relative_to(ROOT)),
                "kind": "table",
                "index": table_idx,
                "style": None,
                "text": " | ".join(cell_texts),
            })

    return blocks


word_blocks = docx_text_blocks(PROJECT_DESCRIPTION_DOCX) + docx_text_blocks(FINAL_REPORT_DOCX)

gliner_labels_de = [
    "Zielgruppe",
    "Projektmaßnahme",
    "Projektziel",
    "Wirkung",
    "Region",
    "Kooperationspartner",
]

relevant_terms = [
    "zielgruppe", "teilnehm", "maßnahme", "projektinhalt", "ziel", "wirkung",
    "beratung", "kurs", "praktika", "lehr", "region", "standort"
]
relevant_blocks = [
    block for block in word_blocks
    if any(term in block["text"].lower() for term in relevant_terms)
]

entities = []
for block in relevant_blocks:
    predictions = gliner_model.predict_entities(block["text"][:3500], gliner_labels_de, threshold=0.35)
    for pred in predictions:
        entities.append({
            "label": pred["label"],
            "text": clean_text(pred["text"]),
            "score": round(float(pred["score"]), 3),
            "source_file": block["source_file"],
            "block_kind": block["kind"],
            "block_index": block["index"],
        })

entities_df = pd.DataFrame(entities).drop_duplicates() if entities else pd.DataFrame()
entities_df.sort_values(["label", "score"], ascending=[True, False]).head(30)

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 550 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 427 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 537 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/pro

,label,text,score,source_file,block_kind,block_index
252,Kooperationspartner,AMS Steiermark,0.820,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,table,6
189,Kooperationspartner,zahlreichen kooperierenden Unternehmen,0.794,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,paragraph,48
148,Kooperationspartner,Rote Kreuz Steiermark,0.747,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,23
109,Kooperationspartner,AMS,0.743,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
104,Kooperationspartner,AMS,0.716,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
210,Kooperationspartner,Projektpartnerschaften,0.709,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,paragraph,67
105,Kooperationspartner,Alpha Nova Jugendcoaching,0.708,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
111,Kooperationspartner,bit,0.675,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
106,Kooperationspartner,bit,0.656,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
223,Kooperationspartner,AMS,0.640,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,table,1


## 8. Select best Word evidence with embeddings

In [9]:
def best_text_blocks(german_query, blocks, top_k=3, min_score=0.35):
    texts = [block["text"] for block in blocks]
    scores = cosine_scores(german_query, texts)
    ranked = sorted(enumerate(scores), key=lambda item: float(item[1]), reverse=True)
    results = []
    for idx, score in ranked[:top_k]:
        score = float(score)
        if score >= min_score:
            block = blocks[idx]
            results.append({
                "score": round(score, 3),
                "source_file": block["source_file"],
                "block_kind": block["kind"],
                "block_index": block["index"],
                "text": block["text"],
            })
    return results


target_group_evidence = best_text_blocks(
    "Zielgruppe des Projekts: Jugendliche und junge Erwachsene mit Migrationshintergrund, die eine Lehre beginnen wollen",
    word_blocks,
    top_k=3,
)

measures_evidence = best_text_blocks(
    "umzusetzende Projektmaßnahmen: Beratung, Vorqualifizierung, Deutschkurs, Workshops, Praktika, Lehrstellenvermittlung",
    word_blocks,
    top_k=3,
)

pd.DataFrame(target_group_evidence + measures_evidence)[["score", "source_file", "block_kind", "block_index", "text"]]

,score,source_file,block_kind,block_index,text
0,0.699,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,table,0,1. Zwischenbericht Die vertraglich vereinbarte...
1,0.647,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,3,Das Ziel der 3-monatigen Qualifizierungsmaßnah...
2,0.647,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,24,Ziel 1: Verbesserung der fachspezifischen Deut...
3,0.736,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,paragraph,43,Im Projektzeitraum wurden in „HandWerk.Zukunft...
4,0.721,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,4,Im Projektzeitraum sollen 6 Maßnahmen umgesetz...
5,0.695,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,107,3.1.3. Erfahrung in der Projektabwicklung


## 9. Build the same SQL-ready fields as the first prototype

In [10]:
def top_entities(label, n=5):
    if entities_df.empty:
        return []
    rows = entities_df[entities_df["label"] == label].sort_values("score", ascending=False).head(n)
    return rows["text"].tolist()


prototype_record = {
    "projektnummer": project_meta["projektnummer"]["value"],
    "projekttitel": project_meta["projekttitel"]["value"],
    "projekttraeger": project_meta["projekttraeger"]["value"],
    "laufzeit_beginn": project_meta["laufzeit_beginn"]["value"],
    "laufzeit_ende": project_meta["laufzeit_ende"]["value"],
    "hauptprojektstandort": project_meta["hauptprojektstandort"]["value"],
    "zielgruppe_gliner_kandidaten": top_entities("Zielgruppe", n=5),
    "projektmassnahmen_gliner_kandidaten": top_entities("Projektmaßnahme", n=8),
    "hauptindikator_name": main_kpi["indikator"],
    "hauptindikator_soll": main_kpi["soll"],
    "hauptindikator_ist": main_kpi["ist"],
    "hauptindikator_erfuellung_prozent": main_kpi["erfuellung_prozent"],
    "zielgruppe_beste_textstelle": target_group_evidence[0]["text"] if target_group_evidence else None,
    "massnahmen_beste_textstelle": measures_evidence[0]["text"] if measures_evidence else None,
}

sql_ready_df = pd.DataFrame([{
    "project_number": prototype_record["projektnummer"],
    "project_name": prototype_record["projekttitel"],
    "organization": prototype_record["projekttraeger"],
    "runtime_start": prototype_record["laufzeit_beginn"],
    "runtime_end": prototype_record["laufzeit_ende"],
    "region": prototype_record["hauptprojektstandort"],
    "main_indicator_name": prototype_record["hauptindikator_name"],
    "main_indicator_soll": prototype_record["hauptindikator_soll"],
    "main_indicator_ist": prototype_record["hauptindikator_ist"],
    "main_indicator_fulfillment_percent": prototype_record["hauptindikator_erfuellung_prozent"],
    "target_group_text": prototype_record["zielgruppe_beste_textstelle"],
    "measures_text": prototype_record["massnahmen_beste_textstelle"],
}])

sql_ready_df

,project_number,project_name,organization,runtime_start,runtime_end,region,main_indicator_name,main_indicator_soll,main_indicator_ist,main_indicator_fulfillment_percent,target_group_text,measures_text
0,042-2024/25,HandWerk.Zukunft – Berufsvorbereitungskurse mi...,,2024-01-01,2025-12-31,Steiermark,Anzahl der Projektteilnehmenden gesamt,180,None,107.8,1. Zwischenbericht Die vertraglich vereinbarte...,Im Projektzeitraum wurden in „HandWerk.Zukunft...


## 10. Evidence and readable output

In [11]:
evidence = {
    "project_meta": project_meta,
    "detected_indicator_table": detected_table_summary,
    "main_kpi": main_kpi,
    "target_group_evidence": target_group_evidence,
    "measures_evidence": measures_evidence,
    "gliner_entities": entities,
}

output_dir = ROOT / "notebooks" / "outputs"
output_dir.mkdir(exist_ok=True)

record_path = output_dir / "042_2024_25_dynamic_prototype_record.json"
evidence_path = output_dir / "042_2024_25_dynamic_extraction_evidence.json"

record_path.write_text(json.dumps(prototype_record, ensure_ascii=False, indent=2), encoding="utf-8")
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Wrote:")
print(record_path.relative_to(ROOT))
print(evidence_path.relative_to(ROOT))

Wrote:
notebooks/outputs/042_2024_25_dynamic_prototype_record.json
notebooks/outputs/042_2024_25_dynamic_extraction_evidence.json


In [12]:
def format_sql_ready_row(df):
    row = df.iloc[0].to_dict()
    lines = ["# SQL-ready extraction result", ""]
    for field, value in row.items():
        if pd.isna(value):
            value = ""
        value = str(value).strip()
        lines.append(f"## `{field}`")
        lines.append(value if value else "_empty_")
        lines.append("")
    return "\n".join(lines)


display(Markdown(format_sql_ready_row(sql_ready_df)))

# SQL-ready extraction result

## `project_number`
042-2024/25

## `project_name`
HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung

## `organization`
_empty_

## `runtime_start`
2024-01-01

## `runtime_end`
2025-12-31

## `region`
Steiermark

## `main_indicator_name`
Anzahl der Projektteilnehmenden gesamt

## `main_indicator_soll`
180

## `main_indicator_ist`
_empty_

## `main_indicator_fulfillment_percent`
107.8

## `target_group_text`
1. Zwischenbericht Die vertraglich vereinbarten Zielgruppen wurden erfolgreich erreicht. Im Fokus standen Jugendliche und junge Erwachsene im Alter von 15 bis 25 Jahren mit Migrationsbiographie. Die Zielgruppe setzt sich wie folgt zusammen (inkl. laufender Kurs): Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive: 13 Personen Asylberechtigte und subsidiär Schutzberechtigte: 9 Personen EU-Bürgerinnen und EU-Bürger: 4 Personen Österreicherinnen und Österreicher mit Migrationshintergrund: 4 Personen Altersstruktur der Teilnehmenden Der Altersdurchschnitt lag sowohl im ersten als auch im zweiten Durchgang bei 19 Jahren. In dem noch laufenden Durchgang sind jedoch mehr 17-jährige Teilnehmende vertreten. Die an den häufigsten vertretenen Altersgruppen sind die 17- sowie die 18- bis 19-Jährigen. Geschlechterverteilung Von den insgesamt 30 Teilnehmenden waren: Männer: 20 (66%) Frauen: 10 (33%) Herkunftsländer Die häufigsten Herkunftsländer der Teilnehmenden sind: Syrien Türkei Bosnien und Herzegowina Afghanistan Österreich 2. Zwischenbericht In der weiteren Projektperiode, das bedeutet auch in der dritten Kursmaßnahme konnte, die im Fördervertrag definierte Zielgruppe, bestehend aus Jugendlichen und jungen Erwachsenen im Alter von 15 bis 25 Jahren mit Migrationsbiografie erfolgreich erreicht werden. Die Zahl der Teilnehmenden erhöhte sich auf insgesamt 47, wobei sich die Zusammensetzung wie folgt verändert hat: Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive: 15 Personen Asylberechtigte und subsidiär Schutzberechtigte: 19 Personen EU-Bürgerinnen und EU-Bürger: 5 Personen Österreicherinnen und Österreicher mit Migrationshintergrund: 8 Personen Es zeigt sich, dass die Anzahl an Asylberechtigten mit subsidiär Schutzberechtigten sowie Drittstaatsangehörigen mit längerfristiger Aufenthaltsperspektive leicht gestiegen ist. Auch die Anzahl an Österreicher:innen mit Migrationshintergrund und EU-Bürger:innen war stärker vertreten. Ein auffälliger Unterschied im Vergleich zu den ersten beiden Kursmaßnahmen zeigte sich in der Altersstruktur. Der Altersdurchschnitt liegt nach wie vor bei rund 19 Jahren, wobei im dritten Kursdurchgang deutlich mehr 16-jährige Jugendliche vertreten waren. Diese jüngere Altersgruppe brachte auch einen höheren Orientierungsbedarf mit, was sich darin zeigte, dass mehr Praktika absolviert wurden, um herauszufinden was einem liegt und was nicht. Die Geschlechterverteilung veränderte sich ebenfalls, so stieg der Anteil der männlichen Teilnehmenden im Vergleich zu den ersten beiden Kursmaßnahmen auf 74,5%, während der Anteil weiblicher Teilnehmender auf 25,5% sank. Die häufigsten Herkunftsländer der Teilnehmenden blieben im Wesentlichen gleich. Angeführt wird die Liste nach wie vor von Syrien (12), Österreich (7), Afghanistan (6), der Türkei (4) und Bosnien und Herzegowina (3), andere (15). 3. Zwischenbericht Die vierte Kursmaßnahme wurde bereits abgeschlossen, die fünfte läuft noch bis 08.08.2025. Im dritten Zwischenbericht werden beide Kurse gemeinsam dargestellt. Die Zielgruppe umfasste 11 Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive, 6 Asyl- bzw. subsidiär Schutzberechtigte, 10 EU-Bürger:innen und 5 Österreicher:innen mit Migrationshintergrund. Der Anteil an Drittstaatsangehörigen und EU-Bürger:innen war höher als in den vorangegangenen Kursen. Der Altersdurchschnitt lag im vierten Kurs bei 18 Jahren, im fünften bei 17 Jahren. Es nahmen erneut überwiegend sehr junge Personen zwischen 16 und 18 Jahren teil. Die Geschlechterverteilung blieb konstant: Im Kurs 4 nahmen 11 Männer und 5 Frauen teil, im Kurs 5 bisher 12 Männer und 4 Frauen. Die häufigsten Herkunftsländer waren die Türkei, Tschetschenien und Afghanistan. 4. Endbericht Die sechste Kursmaßnahme umfasste insgesamt 5 Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive, 6 Asyl- bzw. subsidiär Schutzberechtigte, 7 EU-Bürger:innen und 1 Österreicherin mit Migrationshintergrund. Der Altersdurchschnitt im 6. Kurs lag bei 17 Jahren. Die Geschlechterverteilung blieb konstant: Im 6. Kurs nahmen 14 Männer und 5 Frauen teil. Die häufigsten Herkunftsländer waren Rumänien, Syrien und Ägypten, jeweils mit 3 Teilnehmer:innen. Insgesamt lässt sich festhalten, dass im gesamten Projektverlauf 98 Jugendliche und junge Erwachsene erreicht wurden. Die Teilnehmenden kamen überwiegend aus Drittstaaten bzw. verfügten über einen Asyl- oder subsidiären Schutzstatus. Darüber hinaus nahmen auch EU-Bürger:innen sowie Österreicher:innen mit Migrationshintergrund an den Kursmaßnahmen teil. Die häufigsten Herkunftsländer waren Syrien und Afghanistan. Der Altersdurchschnitt lag über alle Kursmaßnahmen hinweg bei rund 17 bis 19 Jahren, wobei in den späteren Kursdurchgängen vermehrt jüngere Teilnehmende zwischen 16 und 18 Jahren vertreten waren. Die Geschlechterverteilung war durchgehend männlich dominiert, mit einem Anteil von rund 70 % Männern und etwa 30 % Frauen.

## `measures_text`
Im Projektzeitraum wurden in „HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung 6 Kursmaßnahmen (Modul 2) planmäßig umgesetzt. In den jeweils 12 Wochen dauernden Kursmaßnahmen fanden wöchentlich wechselnd fachsprachlicher Unterricht und Workshops statt, begleitet von Mentoring, kursbegleitender Beratung und Unterstützung sowie Reflexions- und Follow-up-Gesprächen, psychosoziale Unterstützung und Nachbetreuung (Module 3 und 4). Der fachsprachliche Unterricht orientierte sich an den Kenntnissen der Teilnehmenden und den Berufswünschen. Die Workshops gliederten sich in die Themenbereiche Berufsbilder, Zeitmanagement, Telefon- und Kommunikationstraining, Bewerbungstraining, berufsspezifische Grundfertigkeiten, Mathematik und Office-Anwendertraining.
